Function: compute window-level vibration features and save a feature table.

Feature strategy:
1. Load window metadata and filtered IMU signals,
2. Reconstruct each window by run_id and time range,
3. Compute time-domain features (mean, std, RMS, peak-to-peak, energy, mean absolute value),
4. Compute frequency-band powers (1-5 Hz, 5-20 Hz, 20-40 Hz),
5. Save engineered features to data/processed/window_features.csv.

The output excludes raw sample sequences and is ready for analysis/modeling.

In [3]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

# Make project root importable whether CWD is repo root or notebooks/
cwd = Path.cwd()
PROJECT_ROOT = cwd if (cwd / 'src').exists() else cwd.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SAMPLE_RATE = 100  # Hz
IMU_COLS = ['ax', 'ay', 'az', 'gx', 'gy', 'gz']

filtered_path = PROJECT_ROOT / 'data' / 'interim' / 'filtered' / 'filtered_dataset.csv'
cleaned_path = PROJECT_ROOT / 'data' / 'interim' / 'cleaned' / 'cleaned_dataset.csv'
windows_path = PROJECT_ROOT / 'data' / 'processed' / 'windowed_metadata.csv'

if filtered_path.exists():
    signal_path = filtered_path
    print('Using filtered dataset for feature extraction')
elif cleaned_path.exists():
    signal_path = cleaned_path
    print('Filtered dataset not found; using cleaned dataset')
else:
    raise FileNotFoundError('No source dataset found. Run 04_cleaning.ipynb and 04b_filtering.ipynb first.')

if not windows_path.exists():
    raise FileNotFoundError(f'Window metadata not found: {windows_path}. Run 05_windowing.ipynb first.')

signals_df = pd.read_csv(signal_path, low_memory=False)
windows_df = pd.read_csv(windows_path, low_memory=False)

missing_imu = [c for c in IMU_COLS if c not in signals_df.columns]
if missing_imu:
    raise KeyError(f'Missing IMU columns in source dataset: {missing_imu}')

required_signal_cols = ['run_id', 't_rel', 'label']
missing_required = [c for c in required_signal_cols if c not in signals_df.columns]
if missing_required:
    raise KeyError(f'Missing required columns in source dataset: {missing_required}')

signals_df = signals_df.sort_values(['run_id', 't_rel']).reset_index(drop=True)

print(f'Signals source: {signal_path}')
print(f'Signals shape: {signals_df.shape}')
print(f'Window metadata shape: {windows_df.shape}')


def band_power_fft(x, fs, f_low, f_high):
    x = np.asarray(x, dtype=float)
    if len(x) < 2:
        return np.nan

    x_centered = x - np.mean(x)
    n = len(x_centered)
    freqs = np.fft.rfftfreq(n, d=1.0 / fs)
    fft_vals = np.fft.rfft(x_centered)
    psd = (np.abs(fft_vals) ** 2) / n

    mask = (freqs >= f_low) & (freqs <= f_high)
    if not np.any(mask):
        return 0.0

    return float(np.trapezoid(psd[mask], freqs[mask]))


def compute_time_features(x):
    x = np.asarray(x, dtype=float)
    if len(x) == 0:
        return {
            'mean': np.nan,
            'std': np.nan,
            'rms': np.nan,
            'ptp': np.nan,
            'energy': np.nan,
            'mean_abs': np.nan,
        }

    return {
        'mean': float(np.mean(x)),
        'std': float(np.std(x)),
        'rms': float(np.sqrt(np.mean(x ** 2))),
        'ptp': float(np.ptp(x)),
        'energy': float(np.sum(x ** 2)),
        'mean_abs': float(np.mean(np.abs(x))),
    }


feature_rows = []

for _, w in windows_df.iterrows():
    run_id = w['run_id']
    t_start = float(w['t_start'])
    t_end = float(w['t_end'])

    window_signal = signals_df[(signals_df['run_id'] == run_id) & (signals_df['t_rel'] >= t_start) & (signals_df['t_rel'] <= t_end)]
    if window_signal.empty:
        continue

    row = {
        'window_id': int(w['window_id']),
        'run_id': run_id,
        'segment_id': int(w['segment_id']),
        'label': w['label'],
        't_start': t_start,
        't_end': t_end,
        'n_samples': int(len(window_signal)),
    }

    for col in IMU_COLS:
        sig = window_signal[col].to_numpy(dtype=float)
        tf = compute_time_features(sig)
        for k, v in tf.items():
            row[f'{col}_{k}'] = v

        row[f'{col}_bandpower_1_5Hz'] = band_power_fft(sig, SAMPLE_RATE, 1.0, 5.0)
        row[f'{col}_bandpower_5_20Hz'] = band_power_fft(sig, SAMPLE_RATE, 5.0, 20.0)
        row[f'{col}_bandpower_20_40Hz'] = band_power_fft(sig, SAMPLE_RATE, 20.0, 40.0)

    acc_mag = np.sqrt((window_signal[['ax', 'ay', 'az']] ** 2).sum(axis=1).to_numpy(dtype=float))
    gyro_mag = np.sqrt((window_signal[['gx', 'gy', 'gz']] ** 2).sum(axis=1).to_numpy(dtype=float))

    for prefix, sig in [('acc_mag', acc_mag), ('gyro_mag', gyro_mag)]:
        tf = compute_time_features(sig)
        for k, v in tf.items():
            row[f'{prefix}_{k}'] = v

        row[f'{prefix}_bandpower_1_5Hz'] = band_power_fft(sig, SAMPLE_RATE, 1.0, 5.0)
        row[f'{prefix}_bandpower_5_20Hz'] = band_power_fft(sig, SAMPLE_RATE, 5.0, 20.0)
        row[f'{prefix}_bandpower_20_40Hz'] = band_power_fft(sig, SAMPLE_RATE, 20.0, 40.0)

    feature_rows.append(row)

features_df = pd.DataFrame(feature_rows).sort_values('window_id').reset_index(drop=True)
if features_df.empty:
    raise RuntimeError('No features were generated. Check window metadata and source signals.')

# Keep only identifiers, label, and engineered features (no raw sample values)
id_cols = ['window_id', 'run_id', 'segment_id', 'label', 't_start', 't_end', 'n_samples']
feature_cols = [c for c in features_df.columns if c not in id_cols]
features_df = features_df[id_cols + feature_cols]

output_dir = PROJECT_ROOT / 'data' / 'processed'
output_dir.mkdir(parents=True, exist_ok=True)
features_path = output_dir / 'window_features.csv'
features_df.to_csv(features_path, index=False)

print(f'Generated window-level vibration features: {features_df.shape}')
print(f'Saved features to: {features_path}')
print('Label distribution:')
print(features_df['label'].value_counts().sort_index().to_string())
print('\nFeature preview:')
print(features_df.head())

Using filtered dataset for feature extraction
Signals source: /Users/pratyush/Desktop/DTU/Bachelor_thesis/BSC_Thesis_intrinsic_sensor_analysis/data/interim/filtered/filtered_dataset.csv
Signals shape: (211705, 14)
Window metadata shape: (1729, 9)
Generated window-level vibration features: (1729, 79)
Saved features to: /Users/pratyush/Desktop/DTU/Bachelor_thesis/BSC_Thesis_intrinsic_sensor_analysis/data/processed/window_features.csv
Label distribution:
label
dry_dirt_track      652
grass               386
muddy_dirt_track    145
smooth_terrain      546

Feature preview:
   window_id                   run_id  segment_id  label  t_start   t_end  \
0          0  log_20260223_142511.490           7  grass   312.23  314.24   
1          1  log_20260223_142511.490          27  grass   317.69  319.70   
2          2  log_20260223_142511.490          28  grass   319.87  321.87   
3          3  log_20260223_142511.490          37  grass   328.83  330.86   
4          4  log_20260223_142511.490  